In [2]:
import matplotlib.pyplot as plt
import time
import numpy as np

# Import pyRTC Core classes and OOPAO interface
from pyRTC import *
from pyRTC.hardware.OOPAOInterface import OOPAOInterface


In [3]:
conf = utils.read_yaml_file("shwfs_OOPAO_config.yaml")

# Split into component sections
confLoop   = conf["loop"]
confWFS    = conf["wfs"]
confWFC    = conf["wfc"]
confPSF    = conf["psf"]
confSlopes = conf["slopes"]

print(confLoop)
print(confWFS)
print(confWFC)
print(confPSF)
print(confSlopes)


In [4]:


shm_names = ["wfs", "wfsRaw", "wfc", "wfc2D", "signal", "signal2D", "psfShort", "psfLong"]
clear_shms(shm_names)


Traceback (most recent call last):
  File "/Users/darrenchailand/opt/anaconda3/envs/ao_research/lib/python3.10/multiprocessing/resource_tracker.py", line 209, in main
    cache[rtype].remove(name)
KeyError: '/wfs'
Traceback (most recent call last):
  File "/Users/darrenchailand/opt/anaconda3/envs/ao_research/lib/python3.10/multiprocessing/resource_tracker.py", line 209, in main
    cache[rtype].remove(name)
KeyError: '/wfs_meta'
Traceback (most recent call last):
  File "/Users/darrenchailand/opt/anaconda3/envs/ao_research/lib/python3.10/multiprocessing/resource_tracker.py", line 209, in main
    cache[rtype].remove(name)
KeyError: '/wfs_gpu_handle'
Traceback (most recent call last):
  File "/Users/darrenchailand/opt/anaconda3/envs/ao_research/lib/python3.10/multiprocessing/resource_tracker.py", line 209, in main
    cache[rtype].remove(name)
KeyError: '/wfsRaw'
Traceback (most recent call last):
  File "/Users/darrenchailand/opt/anaconda3/envs/ao_research/lib/python3.10/multiprocessin

In [5]:
# Create OOPAO simulation interface
sim = OOPAOInterface(conf=conf, param=None)

# Extract references to the simulated hardware
wfs, dm, psf = sim.get_hardware()


In [6]:
from OOPAO.calibration.compute_KL_modal_basis import compute_KL_basis

NUM_MODES = confWFC["numModes"]  # 97
M2C_KL = compute_KL_basis(sim.tel, sim.atm, sim.dm)
dm.setM2C(M2C_KL[:, :NUM_MODES])


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 544 is different from 97)

In [ ]:
slopes = SlopesProcess(conf=confSlopes)
loop   = Loop(conf=confLoop)


In [ ]:
dm.start()
dm.flatten()         # start from flat

wfs.start()
slopes.start()

print("DM OPD shape:", sim.dm.OPD.shape)

psf.start()
dm.flatten()
slopes.plotPupils()  # quick visual sanity check


In [ ]:
# Remove atmosphere for clean IM
sim.removeAtmosphere()

# Optional: reference PSF for Strehl calc (if implemented)
if hasattr(psf, "takeModelPSF"):
    psf.takeModelPSF()

# IM settings from YAML (pokeAmp=9.5e-7, method=push-pull, numItersIM=1)
loop.pokeAmp = confLoop["pokeAmp"]
loop.computeIM()           # blocking; fills IM internally

# Restore atmosphere
sim.addAtmosphere()

# Visualize
plt.imshow(sim.dm.OPD); plt.colorbar(); plt.title("DM OPD (post-IM)"); plt.show()
loop.plotIM()
plt.show()

# (Optional) Save IM/CM for reuse
IM = loop.getIM()
utils.ensure_dir("./cal")
np.save("./cal/IM_SHWFS.npy", IM)
try:
    CM = loop.getCM()
except AttributeError:
    CM = utils.pinv(IM, rcond=1e-3)
np.save("./cal/CM_SHWFS.npy", CM)


In [ ]:
dm.flatten()
time.sleep(1e-2)

loop.setGain(confLoop["gain"])  # 0.1
loop.start()

time.sleep(10)  # run for 10 seconds

loop.stop()
dm.flatten()


In [ ]:
# Science camera PSF (current frame)
plt.imshow(psf.data); plt.colorbar(); plt.title("Science Camera PSF (Current Frame)"); plt.show()

# Normalized PSF from OOPAO
plt.imshow(psf.tel.PSF_norma); plt.colorbar(); plt.title("Normalized PSF (OOPAO)"); plt.show()

# Final DM optical path difference (OPD)
plt.imshow(sim.dm.OPD); plt.colorbar(); plt.title("Final DM Shape (OPD Map)"); plt.show()

# WFS 2D signal (SH slopes maps concatenated)
plt.imshow(sim.wfs.signal_2D); plt.colorbar(); plt.title("WFS 2D Signal (SHWFS)"); plt.show()

# Processed slopes output (SlopesProcess)
plt.imshow(slopes.signal2D.read_noblock()); plt.colorbar(); plt.title("Processed Slopes (2D)"); plt.show()
